In [1]:
import pandas as pd
import numpy as np
import torch
import random
import os
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from tqdm import tqdm

# 1. ФИКСАЦИЯ ВОСПРОИЗВОДИМОСТИ
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# 2. НАСТРОЙКИ
SUBMISSIONS_DIR = "submissions"
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
TIME_LIMIT = 3600 # 1 час. Если есть возможность — ставь 7200 (2 часа)

# 3. ЗАГРУЗКА И РЕКОНСТРУКЦИЯ target_30min (Метод Тисков)
print("Loading data...")
train = pd.read_parquet('data/train_solo_track.parquet')
test = pd.read_parquet('data/test_solo_track.parquet')

def reconstruct_30min_vise(series):
    T = series.values.astype(np.float64)
    n = len(T)
    v30 = np.full(n, np.nan)
    zero_indices = np.where(T == 0)[0]
    if len(zero_indices) > 0:
        for idx in zero_indices:
            v30[idx] = 0
            if idx > 0: v30[idx-1] = 0
        for _ in range(3):
            for i in range(1, n):
                if np.isnan(v30[i]) and not np.isnan(v30[i-1]): v30[i] = max(0, T[i] - v30[i-1])
            for i in range(n-1, 0, -1):
                if np.isnan(v30[i-1]) and not np.isnan(v30[i]): v30[i-1] = max(0, T[i] - v30[i])
    if np.isnan(v30).any():
        s, sign = 0.0, 1
        v0_min, v0_max = 0.0, T[0]
        for i in range(1, n):
            s = T[i] - s
            sign *= -1
            if sign == 1: v0_min, v0_max = max(v0_min, -s), min(v0_max, T[i] - s)
            else: v0_min, v0_max = max(v0_min, s - T[i]), min(v0_max, s)
        v0_best = (v0_min + v0_max) / 2
        V = np.zeros(n); V[0] = v0_best
        for i in range(1, n): V[i] = max(0, T[i] - V[i-1])
        return V
    return v30

print("Reconstructing 30min targets...")
tqdm.pandas()
train['target_30min'] = train.groupby('route_id')['target_1h'].progress_transform(reconstruct_30min_vise)

# 4. ФИЧИ (Временные циклы)
def add_time_features(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    t = df['timestamp'].dt
    df['m_sin'] = np.sin(2 * np.pi * t.minute / 60)
    df['m_cos'] = np.cos(2 * np.pi * t.minute / 60)
    df['h_sin'] = np.sin(2 * np.pi * t.hour / 24)
    df['h_cos'] = np.cos(2 * np.pi * t.hour / 24)
    df['d_sin'] = np.sin(2 * np.pi * t.dayofweek / 7)
    df['d_cos'] = np.cos(2 * np.pi * t.dayofweek / 7)
    
    # ПРИЗНАК РАБОЧЕГО ДНЯ (1 ноября принудительно = 1.0)
    df['is_working_day'] = t.dayofweek.map({0:1, 1:1, 2:1, 3:1, 4:1, 5:0, 6:0}).astype(float)
    df.loc[df['timestamp'].dt.date == pd.to_datetime('2025-11-01').date(), 'is_working_day'] = 1.0
    return df

train = add_time_features(train)
test = add_time_features(test)

time_covs = ['m_sin', 'm_cos', 'h_sin', 'h_cos', 'd_sin', 'd_cos', 'is_working_day']

# 5. ПОДГОТОВКА ДАННЫХ ДЛЯ AG
# Берем последние 14 дней
train_subset = train[train['timestamp'] >= train['timestamp'].max() - pd.Timedelta(days=14)].copy()

ts_train = TimeSeriesDataFrame.from_data_frame(
    train_subset, id_column="route_id", timestamp_column="timestamp"
).sort_index()

ts_test_known = TimeSeriesDataFrame.from_data_frame(
    test[['route_id', 'timestamp'] + time_covs], 
    id_column="route_id", timestamp_column="timestamp"
).sort_index()

# 6. ОБУЧЕНИЕ (BEST QUALITY)
predictor = TimeSeriesPredictor(
    prediction_length=8,
    target="target_30min",
    eval_metric="WAPE",
    freq="30min",
    known_covariates_names=time_covs
)

print("\n🚀 Starting AutoGluon 30min BEST_QUALITY (Raw Mode)...")
predictor.fit(
    ts_train,
    presets="best_quality", 
    time_limit=TIME_LIMIT,
    random_seed=42
)

# 7. ИНФЕРЕНС И ИТЕРАТИВНАЯ СБОРКА 1H ТАРГЕТА
print("\nPredicting and reassembling target_1h...")
preds_30m = predictor.predict(ts_train, known_covariates=ts_test_known).reset_index()

# Словарь последнего факта 30-минутки из трейна
last_30m_heritage = train.groupby('route_id')['target_30min'].last().to_dict()

final_results = []
for rid in tqdm(range(1000), desc="Reassembling routes"):
    # Прогнозы модели (8 точек по 30 минут)
    route_preds = preds_30m[preds_30m['item_id'] == rid]['mean'].values
    # То самое значение 10:30 утра, которое входит в первое часовое окно теста
    heritage = last_30m_heritage[rid]
    
    # Собираем часовое скользящее окно:
    # 11:00 = pred(11:00) + fact(10:30)
    p1h_first = route_preds[0] + heritage
    # Все остальные шаги теста (11:30 - 14:30)
    p1h_others = route_preds[1:] + route_preds[:-1]
    
    combined_p1h = np.concatenate([[p1h_first], p1h_others])
    
    # Мапим на ID теста
    test_rows = test[test['route_id'] == rid].sort_values('timestamp')
    test_ids = test_rows['id'].values
    
    for i in range(8):
        final_results.append({
            'id': test_ids[i], 
            'y_pred': max(0.0, combined_p1h[i])
        })

# 8. СОХРАНЕНИЕ БЕЗ КАЛИБРОВКИ
sub_df = pd.DataFrame(final_results).sort_values('id')
out_path = 'submissions/081_30min_BEST_QUALITY_RAW.csv'
sub_df.to_csv(out_path, index=False)

print(f"\n✅ Done! Raw Best Quality sub saved to {out_path}")
print(f"Predicted Mean Volume: {sub_df['y_pred'].mean():.2f}")
print(f"Check: how far is it from our golden 279k? {((sub_df['y_pred'].mean()/279300)-1)*100:+.2f}%")

Loading data...
Reconstructing 30min targets...


100%|██████████| 1000/1000 [00:16<00:00, 60.94it/s]
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\Николай\PycharmProjects\WBShipment\AutogluonModels\ag-20260406_151825'



🚀 Starting AutoGluon 30min BEST_QUALITY (Raw Mode)...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.1
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          20
Pytorch Version:    2.7.1+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 6.00/6.00 GB
Total GPU Memory:   Free: 6.00 GB, Allocated: 0.00 GB, Total: 6.00 GB
GPU Count:          1
Memory Avail:       8.09 GB / 23.71 GB (34.1%)
Disk Space Avail:   105.72 GB / 476.13 GB (22.2%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': '30min',
 'hyperparameters': 'default',
 'known_covariates_names': ['m_sin',
                            'm_cos',
                            'h_sin',
                            'h_cos',
                            'd_sin',
                            'd_cos',
                            'is_working_day'],
 'num_val_windows': 'auto',
 'prediction_length': 8,
 'quantile_levels': [0.1


Predicting and reassembling target_1h...


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Reassembling routes: 100%|██████████| 1000/1000 [00:00<00:00, 1399.86it/s]


✅ Done! Raw Best Quality sub saved to submissions/081_30min_BEST_QUALITY_RAW.csv
Predicted Mean Volume: 258867.92
Check: how far is it from our golden 279k? -7.32%
